In [31]:
from ingestion.transcript_news import  get_earnings_call_transcript, get_news_sentiment,keygen
import pandas as pd
import yfinance as yf
from itertools import product
import time
import json
from datetime import datetime
import duckdb
from dotenv import load_dotenv
import os
load_dotenv()



True

In [40]:
# Read the entire encrypted dataset
con = duckdb.connect()
con.sql(f"PRAGMA add_parquet_key('main_key', {os.getenv('DUCKDB_KEY')});")

df = con.sql("""
    SELECT * FROM read_parquet(
        'ingestion/data/transcripts/symbol=*/*.parquet',
        encryption_config = {footer_key: 'main_key'},
        hive_partitioning = true
    )
""").to_df()

df

,fiscalDateEnding,reportedDate,time_from,av_quarter,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,transcript,symbol
0,2026-03-31,2026-04-30,20260430T0000,2026Q1,2.01,1.94,0.07,3.6082,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':...",AAPL
1,2025-12-31,2026-01-29,20260129T0000,2025Q4,2.84,2.67,0.17,6.367,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':...",AAPL
2,2025-09-30,2025-10-30,20251030T0000,2025Q3,1.85,1.77,0.08,4.5198,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':...",AAPL
3,2025-06-30,2025-07-31,20250731T0000,2025Q2,1.57,1.43,0.14,9.7902,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':...",AAPL
4,2025-03-31,2025-05-01,20250501T0000,2025Q1,1.65,1.62,0.03,1.8519,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':...",AAPL
...,...,...,...,...,...,...,...,...,...,...,...
492,2015-02-28,2015-03-17,20150317T0000,2015Q1,0.68,0.68,0,0,post-market,"[{'speaker': 'Ken Bond', 'title': 'IR', 'conte...",ORCL
493,2014-11-30,2014-12-17,20141217T0000,2014Q4,0.69,0.68,0.01,1.4706,post-market,"[{'speaker': 'Ken Bond', 'title': 'VP of Inves...",ORCL
494,2014-08-31,2014-09-18,20140918T0000,2014Q3,0.62,0.64,-0.02,-3.125,post-market,"[{'speaker': 'Ken Bond', 'title': 'Vice Presid...",ORCL
495,2014-05-31,2014-06-19,20140619T0000,2014Q2,0.92,0.95,-0.03,-3.1579,post-market,"[{'speaker': 'Ken Bond', 'title': 'Vice Presid...",ORCL


In [23]:
query = """
SELECT symbol,count(*) as transcript_count  FROM read_csv('ingestion/data/transcripts/*.csv')

group by symbol


"""

df = duckdb.query(query).to_df()
df

,symbol,transcript_count
0,AAPL,50
1,MU,50
2,CSCO,50
3,NVDA,49
4,LRCX,50
5,MSFT,50
6,AVGO,49
7,ORCL,49
8,INTC,50
9,AMD,50


In [24]:
query = """
SELECT *


FROM read_csv('ingestion/data/transcripts/*.csv')



"""

df = duckdb.query(query).to_df()
df

,symbol,fiscalDateEnding,reportedDate,time_from,av_quarter,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,transcript
0,AAPL,2026-03-31,2026-04-30,20260430T0000,2026Q1,2.01,1.94,0.07,3.6082,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':..."
1,AAPL,2025-12-31,2026-01-29,20260129T0000,2025Q4,2.84,2.67,0.17,6.3670,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':..."
2,AAPL,2025-09-30,2025-10-30,20251030T0000,2025Q3,1.85,1.77,0.08,4.5198,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':..."
3,AAPL,2025-06-30,2025-07-31,20250731T0000,2025Q2,1.57,1.43,0.14,9.7902,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':..."
4,AAPL,2025-03-31,2025-05-01,20250501T0000,2025Q1,1.65,1.62,0.03,1.8519,post-market,"[{'speaker': 'Suhasini Chandramouli', 'title':..."
...,...,...,...,...,...,...,...,...,...,...,...
492,ORCL,2015-02-28,2015-03-17,20150317T0000,2015Q1,0.68,0.68,0.00,0.0000,post-market,"[{'speaker': 'Ken Bond', 'title': 'IR', 'conte..."
493,ORCL,2014-11-30,2014-12-17,20141217T0000,2014Q4,0.69,0.68,0.01,1.4706,post-market,"[{'speaker': 'Ken Bond', 'title': 'VP of Inves..."
494,ORCL,2014-08-31,2014-09-18,20140918T0000,2014Q3,0.62,0.64,-0.02,-3.1250,post-market,"[{'speaker': 'Ken Bond', 'title': 'Vice Presid..."
495,ORCL,2014-05-31,2014-06-19,20140619T0000,2014Q2,0.92,0.95,-0.03,-3.1579,post-market,"[{'speaker': 'Ken Bond', 'title': 'Vice Presid..."


In [41]:
df['transcript'].explode('transcript').head()

0    {'speaker': 'Suhasini Chandramouli', 'title': ...
1    {'speaker': 'Timothy D. Cook', 'title': 'CEO',...
2    {'speaker': 'Kevan Parekh', 'title': 'CFO', 'c...
3    {'speaker': 'Suhasini Chandramouli', 'title': ...
4    {'speaker': 'Amit Daryanani', 'title': 'Analys...
Name: transcript, dtype: object